In [ ]:
"""

Prepare reference obs for CESMEP

"""

In [47]:
import xarray as xr
import gsw

In [2]:
%matplotlib qt5

QStandardPaths: error creating runtime directory '/run/user/2784' (Permission denied)


In [8]:
inputpath_obs = '/data/cburgard/EVAL_IPSLCM/REFERENCE_OBS/'
#T_obs = xr.open_dataset(inputpath_obs + 'woa23_decav_t_clim_tan_04.nc',decode_times=False)
#S_obs = xr.open_dataset(inputpath_obs + 'woa23_decav_s_clim_san_04.nc',decode_times=False)
T_obs = xr.open_dataset(inputpath_obs + 'woa23_decav_t00_04.nc',decode_times=False)
S_obs = xr.open_dataset(inputpath_obs + 'woa23_decav_s00_04.nc',decode_times=False)

In [9]:
#mask_ocean = S_obs['s_an'].isel(time=0).drop('time') > 0
mask_ocean = S_obs['s_an'].isel(time=0).drop('time') > 0

In [10]:
# pour tous les points
vert_diff_plus_all = (mask_ocean - mask_ocean.shift(depth=1)).isel(depth=range(1,len(mask_ocean.depth)))
vert_diff_minus_all = (mask_ocean - mask_ocean.shift(depth=-1)).isel(depth=range(1,len(mask_ocean.depth)))

In [33]:
# dernier point d'océan avant le fond - somme de toutes les profondeurs où la différence est positive (donc la transition océan-fond)
bot_depth_all = (mask_ocean.depth * vert_diff_minus_all).where(vert_diff_minus_all > 0).sum('depth').astype('float')

In [34]:
max_depth = mask_ocean.depth.max().values
bot_depth_all = bot_depth_all.where((mask_ocean.sum('depth') != len(mask_ocean.depth)),max_depth)

In [37]:
Tbot = T_obs['t_an'].sel(depth=bot_depth_all).where(bot_depth_all > 0)
Sbot = S_obs['s_an'].sel(depth=bot_depth_all).where(bot_depth_all > 0)

In [46]:
#Tbot.rename('T_bottom').to_netcdf(inputpath_obs + 'T_bottom_WOA23_clim_1991_2020_deepest1500.nc')
#Sbot.rename('S_bottom').to_netcdf(inputpath_obs + 'S_bottom_WOA23_clim_1991_2020_deepest1500.nc')
Tbot.drop('depth').squeeze().drop('time').rename('T_bottom').to_netcdf(inputpath_obs + 'T_bottom_WOA23_1991_2020_annualmean.nc')
Sbot.drop('depth').squeeze().drop('time').rename('S_bottom').to_netcdf(inputpath_obs + 'S_bottom_WOA23_1991_2020_annualmean.nc')

In [48]:
rhobot = gsw.density.sigma0(Sbot, Tbot)

In [52]:
rhobot.drop('depth').squeeze().drop('time').rename('rhopot_bottom').to_netcdf(inputpath_obs + 'rhopot_bottom_WOA23_1991_2020_annualmean.nc')

In [51]:
rhobot.plot(vmin=27.7,vmax=28.1)

In [ ]:
# Now need to write it out to file to be used as a reference obs

In [ ]:
### FROM JACQUEMINE, NOT NEEDED ANYMORE

In [ ]:
def list_sizes(ds):
    sizes = ds.sizes
    print(sizes)
    integer_values_list = [value for value in sizes.values() if isinstance(value, int)]
    return integer_values_list

In [ ]:
values_list_obs = list_sizes(ds_obs)

In [ ]:
def calculate_bottom_obs(ds, var_t, var_s, integer_values_list):
    n_lat, _, n_lon, max_depth = integer_values_list

    tbot = xr.DataArray(np.full((len(ds.lat), len(ds.lon)), np.nan, dtype='float32'),
                        dims=("lat", "lon"), coords={'lat': ds.lat, 'lon': ds.lon})
    sbot = tbot.copy()
    dbot = tbot.copy()

    for i in range(n_lat):
        for j in range(n_lon):
            temp_profile = ds[var_t].isel(lat=i, lon=j).values
            sal_profile = ds[var_s].isel(lat=i, lon=j).values
            depth_profile = ds.depth.values

            nan_indices = np.where(np.isnan(temp_profile))[0]

            if nan_indices.size == 0:
                idx = max_depth - 1
            elif nan_indices[0] == 0:
                tbot[i, j] = temp_profile[0]
                sbot[i, j] = sal_profile[0]
                dbot[i, j] = np.nan
                continue
            else:
                idx = nan_indices[0] - 1

            tbot[i, j] = temp_profile[idx]
            sbot[i, j] = sal_profile[idx]
            dbot[i, j] = depth_profile[idx]

    ds["TBOT"] = tbot
    ds["SBOT"] = sbot
    ds["DBOT"] = dbot

    return ds

In [ ]:
dataset = calculate_bottom_obs(ds_obs, 't_an', 's_an', values_list_obs)